In [267]:
import numpy as np
import pandas as pd

Previous Notebook: `\Cleaned Data\2023-03-04 Joining DFs`

Import the csv containing the citaions as `papers_df` and the data frame containing the query author information as `query_df`

In [268]:
papers_df = pd.read_csv(r"G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data\2023-03-04 Joining DFs_join_Df2.csv", low_memory = True)

Here is `papers_df`

In [269]:
papers_df.head()

,Cites,Title,Year,Source,GSRank,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth,ArticleType,SJR,Hindex,SourceCountry,SourceRegion
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,4186,523.25,523,8,0,d shi,v adinolfi,r comin,journal,14.589,1229,United States,Northern America
1,1951,efficient and stable solution-processed planar...,2017.0,science,4,1951,325.17,279,7,0,h tan,a jain,o voznyy,journal,14.589,1229,United States,Northern America
2,1756,homogeneously dispersed multimetal oxygen-evol...,2016.0,science,7,1756,250.86,251,7,0,b zhang,x zheng,o voznyy,journal,14.589,1229,United States,Northern America
3,1343,co2 electroreduction to ethylene via hydroxide...,2018.0,science,11,1343,268.60,224,6,0,ct dinh,t burdyny,mg kibria,journal,14.589,1229,United States,Northern America
4,1168,what would it take for renewably powered elect...,2019.0,science,14,1168,292.00,195,6,0,p de luna,c hahn,d higgins,journal,14.589,1229,United States,Northern America


In [270]:
query_df = pd.read_csv(r"G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Raw Data\2023-03-03 Export\author_PoPMetrics_V2.2.csv", low_memory = True)

Here is `query_df`

In [271]:
query_df.head()

,id,Query,Source,Papers,Citations,Years,Cites_Year,Cites_Paper,Cites_Author,Papers_Author,...,g_coverage,star_count,year_first,year_last,ECC,acc1,acc2,acc5,acc20,hA
0,0,Edward Sargent - University of Toronto,Google Scholar Profile,1000,119190,27,4414.44,119.19,22989.19,225.13,...,88.1,383,1996,2023,119190,751,665,517,247,69
1,1,Norbert Koch - Institut für Physik &IRIS Adler...,Google Scholar Profile,560,26197,26,1007.58,46.78,5580.69,106.68,...,83.4,76,1997,2023,26197,337,285,166,30,24
2,2,"Dieter Neher - University of Potsdam, Institut...",Google Scholar Profile,559,36923,87,424.40,66.05,7764.41,115.04,...,86.3,122,1936,2023,36923,329,274,183,65,31
3,3,"Daniel Duprez - IC2MP, Université de Poitiers,...",Google Scholar Profile,503,18350,52,352.88,36.48,5375.41,153.68,...,84.3,24,1971,2021,18350,203,148,68,5,14
4,4,"Sergey Makarov - Professor, Head of Laboratory",Google Scholar Profile,414,6104,35,174.40,14.74,1033.45,83.01,...,72.4,38,1988,2023,6104,175,146,78,11,15


Let's first edit the names in `Query` so that they match the format given to the author names in `paper_df`. Assigning those values to the new column `query_author`.

In [272]:
import re

def edit_names(name):
    # remove any non-UTF-8 characters using encode method
    name = name.encode('ascii', 'ignore').decode('ascii')
    
    # remove any parenthess, hyphans and periodsusing regenx
    name = re.sub(r'[\(\)-\.,]', '', name).strip()
    
    # remove double spaces
    name = re.sub(r'  ', '', name).strip()
    
    # Use re to remove commas and anything after them
    name = re.sub(r'\,.*', '', name).strip()
    
    # Use re to remove title: Dr
    name = re.sub(r'^Dr\.?\s*', '', name).strip()
    
    # Use re to remove title: phd
    name = re.sub(r'^PhD\s*', '', name).strip()
    
    # Use re to remove phd
    name = re.sub(r'Ph\.?D\.?.*', '', name).strip()
    
    # Use re to remove title: Proff.
    name = re.sub(r'^PROFF\.?\s*', '', name).strip()
    
    # Remove the title: 'jr'
    name = re.sub(r'jr\.?s*', '', name).strip()
    
    # Rmove the title 'Junior'
    name = re.sub(r'junior', '', name).strip()
    
    # Remove 'equal contribution'
    name = re.sub(r'equal contribution', '', name).strip()
    
    # Remove 'ltslmw'
    name = re.sub(r'lts', '', name).strip()
    
    # Remove 'lmw'
    name = re.sub(r'lmw', '', name).strip()
    
    # Remove 'orcid'
    name = re.sub(r'ORCID:0000000160401920', '', name)
    
    # Join last names with `van` in them 
    if 'van' in name:
        new_split = name.rsplit(' ')
        if len(new_split) > 2:
            name = new_split[0] + ' ' + ''.join(new_split[1:])
        else:
            pass
    
    # Join last names with articles in them
    elif 'de' in name or 'dos' or 'da' in name or 'do' in name:
        new_split = name.rsplit(' ')
        if 'del' in new_split:
            article = 'del'
        elif 'dos' in new_split:
            article = 'dos'
        elif 'da' in new_split:
            article = 'da'
        elif 'do' in new_split:
            article = 'do'
        elif 'de' in new_split:
            article = 'de'
        else:
            article = 'none'
        if len(new_split) > 2:
            new_split = np.asarray(new_split)
            de_idx, = np.where(new_split == article) # where the article is
            if len(de_idx) != 0:
                first = ' '.join(new_split[:de_idx[0]]) # first is the name up to the article
                last = ''.join(new_split[de_idx[0]:]) # last is the name from the article onward
                name = ' '.join([first, last])
            
    # makes sure the name is in lower case, do a final strip
    return name.lower().strip()

In [273]:
def firstlast_names(input_name):
    
    if isinstance(input_name, str):
        
        # Pass name into primary format editor
        name = edit_names(input_name)
        
        # split the string into a list
        split_col = [i for i in name.rsplit(' ') if isinstance(i, str)]
        
        if len(split_col) == 1:
            split_col = [edit_names(i) for i in input_name.rsplit('.') if isinstance(i, str) and edit_names(i) != '']
        
        if len(split_col) == 1:
            # still 1, just return the name
            return name
            
        if len(split_col[0]) == 1:
            # Example: 
            # >> split_col = ['k', 'yokuda']
            # >> first_mid_initial = ['k']
            first_mid_initial = split_col[0]
        
        # Check if the first inital actually contains a middle inital in the form of a string with length = 2
        # (string with length > 2 could mean a full first name is written out)
        
        #If the first inital has actually two letters
        elif len(split_col[0]) == 2 and len(split_col) < 3: 
            # Example: 
            # >> split_col = ['km', 'yokuda']
            # >> first_mid_initial = ['km']
            first_mid_initial = split_col[0][:2]
            
        else:
            # If there is a middle name
            if len(split_col) >= 4: 
                
                if [i for i in split_col[0]] == [split_col[1][0], split_col[2][0], split_col[3][0]]:
                    # Example: 
                    # >> split_col = ['kmy', 'kelly', 'marie', 'yokuda'] 
                    # >> first_mid_initial = 'km'
                    first_mid_initial = split_col[1][0] + split_col[2][0]
                
                else:
                    # Example:
                    # >> split_col = ['kelly', 'maire', 'yoshi', 'yokuda']
                    # >> first_mid_initial = 'kmy'
                    first_mid_initial = split_col[0][0] + split_col[1][0] + split_col[2][0]
            
            elif len(split_col) == 3:
                # Example:
                # >> split_col = ['kelly', 'maire', 'yokuda']
                # >> first_mid_initial = 'km'
                first_mid_initial = split_col[0][0] + split_col[1][0]
            
            elif len(split_col) == 2:
                # To help give more spicific identiy to asian names
                
                # Option 1:
                # split_col = ['jingying', 'liu']
                # first_mid_initial = 'jy'
                match = re.match(r'(.ing)(.ing)', split_col[0])
                if match:
                    first_mid_initial = match.group(1)[0] + match.group(2)[0]
                elif match == None:
                    # Option 2:
                    # >> split_col = ['jingjin' 'dong']
                    # >> first_mid_initial = jj
                    match = re.match(r'(.ing)(.in)', split_col[0])
                    if match:
                        first_mid_initial = match.group(1)[0] + match.group(2)[0]
                    elif match == None:
                        # Option 3:
                        # >> split_col = ['zhongyao', 'jiang']
                        # >> first_mid_initial = zy
                        match = re.match(r'(..ong)(.ao)', split_col[0])
                        if match:
                            first_mid_initial = match.group(1)[0] + match.group(2)[0]
                        # Option 4:
                        # >> split_col = ['xuemei', 'li']
                        # >> first_mid_initial = 'xuemei'
                        if (split_col[0][0] == 'x' or split_col[0][0] == 'y') and (split_col[1][0] == 'z' or split_col[1][0] == 'l'):
                            first_mid_initial = split_col[0]
                        elif match == None:
                            # No middle name
                            # Example:
                            # >> split_col = ['kelly', 'yokuda']
                            # >> first_mid_initial = 'k'
                            first_mid_initial = split_col[0][0]

        
        last_name = split_col[-1]
        return ' '.join([first_mid_initial, last_name]).strip()
    
    else:
        return None

In [274]:
def edit_papers_names(row):
    return [firstlast_names(i) for i in row]

def edit_query_names(row):
    split_col = [i.strip() for i in row['Query'].rsplit(' - ')]
    name = split_col[0]
    ret_val = firstlast_names(name)
    return ret_val

In [275]:
query_df = query_df.assign(query_author = query_df.apply(edit_query_names, axis = 1))

In [276]:
query_df.sample(3)

,id,Query,Source,Papers,Citations,Years,Cites_Year,Cites_Paper,Cites_Author,Papers_Author,...,star_count,year_first,year_last,ECC,acc1,acc2,acc5,acc20,hA,query_author
424,424,Akriti - Lam Research | Purdue University,Google Scholar Profile,15,761,5,152.2,50.73,106.24,2.62,...,5,2018,2022,761,12,11,8,3,7,akriti
286,286,Samia Kausar - University of Gujrat,Google Scholar Profile,30,234,6,39.0,7.80,35.09,4.92,...,1,2017,2023,234,20,17,6,0,5,s kausar
264,264,Rhiannon Kennard - Faraday Research Fellow at ...,Google Scholar Profile,33,1064,10,106.4,32.24,139.26,7.06,...,7,2013,2022,1064,20,17,12,2,8,r kennard


In [277]:
result = query_df['query_author'].value_counts().sum() == query_df['Query'].value_counts().sum()
print('Are the number of unqiue Queries maintained?\t', result)

Are the number of unqiue Queries maintained?	 True


Make a `sub_query_df`

In [278]:
sub_query_df = query_df.loc[:, ['id', 'query_author']]

In [279]:
sub_query_df.head()

,id,query_author
0,0,e sargent
1,1,n koch
2,2,d neher
3,3,d duprez
4,4,s makarov


Perform the same name formatting process of the names in the `first_auth`, `second_auth` and `third_auth` columns in `papers_df`.

In [280]:
papers_df.loc[:, 'first_auth':'third_auth'] = papers_df.loc[:, 'first_auth':'third_auth'].apply(edit_papers_names)

In [281]:
papers_df.loc[:, 'first_auth':'third_auth'].sample(3)

,first_auth,second_auth,third_auth
8765,p papagiorgis,l protesescu,mv kovalenko
6941,j yin,gh ahmed,om bakr
7036,w peng,c aranda,om bakr


Make a `sub_papers_df`

In [282]:
sub_papers_df = papers_df.loc[:, ['query_id', 'Title', 'first_auth', 'second_auth', 'third_auth']]

In [283]:
sub_papers_df.sample(3)

,query_id,Title,first_auth,second_auth,third_auth
14320,118,origin of high thermoelectric performance with...,z li,s han,y pan
10981,175,preserving the stoichiometry of triple-cation ...,o telschow,m albaladejosiguan,l merten
3141,0,efficient electrocatalytic conversion of carbo...,y pang,j li,z wang


Join the data frames using inner join

In [284]:
join_df = pd.merge(left = sub_papers_df, right = sub_query_df, left_on = 'query_id', right_on = 'id').drop('id', axis = 1)
join_df.head()

,query_id,Title,first_auth,second_auth,third_auth,query_author
0,0,low trap-state density and long carrier diffus...,d shi,v adinolfi,r comin,e sargent
1,0,efficient and stable solution-processed planar...,h tan,a jain,o voznyy,e sargent
2,0,homogeneously dispersed multimetal oxygen-evol...,b zhang,x zheng,o voznyy,e sargent
3,0,co2 electroreduction to ethylene via hydroxide...,ct dinh,t burdyny,mg kibria,e sargent
4,0,what would it take for renewably powered elect...,p deluna,c hahn,d higgins,e sargent


In [285]:
result = papers_df.shape[0] == join_df.shape[0]
print('Are the number of rows maintained?\t', result)

Are the number of rows maintained?	 True


Let's check the number of unique authors.

In [286]:
get_unique_auth(join_df)

There are 16397 unique author names


In [287]:
from difflib import SequenceMatcher

def get_unique_auth(df):
    
    author_list = np.concatenate(df.loc[:, 'first_auth':'third_auth'].values)
    auth_ser = pd.Series(author_list)
    
    og_names = auth_ser.unique().shape[0]
    print(f"There are {og_names} unique author names")
    

def all_same(itemlist):
    '''
    Dermines if all the elements in a list are the same.
    
    '''
    return all(x == itemlist[0] for x in itemlist)

def pull_name3(row, v, col = None):
    
    '''
    For each title in the data set, check if there is more than one citation. If there is, check
    if the first authors names are formatted in the same way. If not, print the variation of the 
    name that is the longest.
    '''
    if v == 'v1':
        og_vals = row
    elif v == 'v2':
        og_vals = row[col].values
    
    #print(og_vals)
    new_vals = np.unique([i for i in og_vals if isinstance(i, str)])
    
    if len(new_vals) > 1:
        replace_dic = {}
        if all_same(new_vals) == False:
            last_names = set([i.rsplit(' ')[-1] for i in new_vals]) # make a list of the last names
            first_initial = set([i.rsplit(' ')[0] for i in new_vals]) # make a list of the first initial
            if len(last_names) == 1 and len(first_initial) == 1: # Check if the last names are all the same.
                print(new_vals)
                top_length = 0
                top_name = ''
                for i in new_vals:
                    if isinstance(i, str):
                        # Checking the series of value counts
                        name, cts = auth_val[auth_val.index == i].index, auth_val[auth_val.index == i].values
                        print(name)
                        length = len(name[0])
                        if length > top_length:
                            top_length = length
                            top_name = name
                    else:
                        pass
                for i in new_vals:
                    if isinstance(i, str):
                        replace_dic[i] = top_name[0]
                    else:
                        pass
                    
            else: # if the last names aren't the same
                for i in range(0, len(new_vals)):
                    for j in range(i+1, len(new_vals)):
                        a = new_vals[i]
                        b = new_vals[j]

                        pass_letters =  ['x', 'y', 'z']
                        ## These are letters that many asian names that would be considered > 80% similar by the seqeunce matcher would start with, but the names are meant to be different

                        if a != b and a[0] == b[0] and a[0] not in pass_letters and b[0] not in pass_letters: # the names aren't the same, but they start with the same letter
                            r = SequenceMatcher(None, a, b).ratio()

                            if r > 0.80: # if the names are at least 85% similar
                                name = [a,b]
                                name.sort(reverse = False, key = len)
                                replace_dic.update({name[0] : name[1]})
                
            if len(replace_dic) != 0: #if replace_dic is not empty
                return replace_dic
            else:
                return False
        else:
            return False
    else:
        return False

First apply on groups by `Title`

In [288]:
## Initiate Replacement dictionary
replace_dic = {}

# Loop over each author column
for i in ['first_auth', 'second_auth', 'third_auth', 'query_author']:
    return_vals = join_df.groupby('Title').apply(pull_name3, v = 'v2', col = i).values
    for k in return_vals[return_vals != False]:
        replace_dic.update(k) 

replace_dic

{'j baena': 'jc baena',
 'dw quilettes': 'dw dequilettes',
 'j arenas': 'jn arenas',
 'e tiguntseva': 'ey tiguntseva',
 'gh ahmed': 'gh ahmeda',
 'h adl': 'hp adl',
 'i surez': 'i suarez',
 'v kambersk': 'v kambersky',
 'a ltoublon': 'a letoublon',
 'f dearquer': 'fg dearquer',
 'p rodrguezcant': 'pj rodrguezcant',
 'j hervs': 'j hervas',
 'm saidaminov': 'mi saidaminov',
 'r patterson': 'rj patterson',
 'g muozmatutano': 'g munozmatutano',
 'g ndjawa': 'gn ndjawa',
 'f peacamargo': 'f penacamargo'}

Let's replace the `join_df` with these values.

In [289]:
join_df = join_df.replace(replace_dic)
get_unique_auth(join_df)

There are 16380 unique author names


Now let's group by row of `first_auth`, `second_auth` and `third_auth`

In [290]:
## Initiate Replacement dictionary
replace_dic = {}

for i in range(join_df.shape[0]):
    vals = join_df.loc[i, 'first_auth':].values
    return_vals = pull_name3(vals, v = 'v1')
    if return_vals != False:
        replace_dic.update(return_vals)

replace_dic

{'e sargent': 'eh sargent',
 'a kiani': 'ar kirmani',
 's mahshid': 'ss mahshid',
 'o bakr': 'om bakr',
 'a de': 'a dey',
 'd lidzey': 'dg lidzey',
 't wu': 'tz wu',
 'e aydil': 'es aydil',
 'g xing': 'gc xing',
 'g ding': 'g xing',
 'j berry': 'jj berry',
 'j gong': 'j tong',
 'k weber': 'kj weber',
 'l wan': 'l wang',
 'j kim': 'jy kim',
 's lee': 'sj lee',
 'jw kim': 'jy kim',
 'h kim': 'hb kim',
 'h cho': 'h choi',
 'j luther': 'jm luther',
 'm saba': 'mi saba',
 'm bodnarchuk': 'mi bodnarchuk',
 'd dirin': 'dn dirin',
 'g eperon': 'ge eperon',
 'ej jurezprez': 'ej juarezperez',
 'j lee': 'jh lee',
 'e gaulding': 'ea gaulding',
 'p groszewicz': 'pb groszewicz',
 'nd marco': 'n demarco',
 'e wong': 'el wong',
 'c soe': 'cmm soe',
 'j snaider': 'jm snaider',
 'r scheidt': 'ra scheidt',
 'm vruini': 'm vrucinic',
 'pg fernndez': 'pg fernandez',
 'p fernndez': 'pg fernandez',
 'p fernandez': 'pg fernandez',
 'tj savenije': 'tj savenjie',
 't savenije': 'tj savenije',
 'a mclaughlin': '

Let's replace the `join_df` with these values.

In [291]:
join_df = join_df.replace(replace_dic)
get_unique_auth(join_df)

There are 16250 unique author names


We end up with 16,254 unique authors from the original 16,397.

In [292]:
print('Thats {:0.3g} of unqiue authors introduced per papers!'.format(papers_df.shape[0]/16254))

Thats 1.39 of unqiue authors introduced per papers!


In [318]:
join_df.head()

,query_id,Title,first_auth,second_auth,third_auth,query_author
0,0,low trap-state density and long carrier diffus...,d shi,v adinolfi,r comin,eh sargent
1,0,efficient and stable solution-processed planar...,h tan,a jain,o voznyy,eh sargent
2,0,homogeneously dispersed multimetal oxygen-evol...,bo zhang,x zheng,o voznyy,eh sargent
3,0,co2 electroreduction to ethylene via hydroxide...,ct dinh,t burdyny,mg kibria,eh sargent
4,0,what would it take for renewably powered elect...,p deluna,c hahn,d higgins,eh sargent


Let's overwrite the `first_auth`, `second_auth`, `third_auth` columns `papers_df` with `join_df` with the addition of the `query_author` column.

In [327]:
papers_df.loc[:, ['first_auth', 'second_auth', 'third_auth']] = join_df.loc[:, ['first_auth', 'second_auth', 'third_auth']]
papers_df = pd.concat([papers_df, join_df.loc[:, 'query_author']], axis = 1)

In [328]:
papers_df.head()

,Cites,Title,Year,Source,GSRank,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth,ArticleType,SJR,Hindex,SourceCountry,SourceRegion,query_author
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,4186,523.25,523,8,0,d shi,v adinolfi,r comin,journal,14.589,1229,United States,Northern America,eh sargent
1,1951,efficient and stable solution-processed planar...,2017.0,science,4,1951,325.17,279,7,0,h tan,a jain,o voznyy,journal,14.589,1229,United States,Northern America,eh sargent
2,1756,homogeneously dispersed multimetal oxygen-evol...,2016.0,science,7,1756,250.86,251,7,0,bo zhang,x zheng,o voznyy,journal,14.589,1229,United States,Northern America,eh sargent
3,1343,co2 electroreduction to ethylene via hydroxide...,2018.0,science,11,1343,268.60,224,6,0,ct dinh,t burdyny,mg kibria,journal,14.589,1229,United States,Northern America,eh sargent
4,1168,what would it take for renewably powered elect...,2019.0,science,14,1168,292.00,195,6,0,p deluna,c hahn,d higgins,journal,14.589,1229,United States,Northern America,eh sargent


Save the new data frames

In [329]:
import os

clean_path = 'G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data'
clean_file = os.path.join(clean_path, '2023-03-10 Clean Names_paper_df.csv')
papers_df.to_csv(clean_file, index_label = False)

In [330]:
clean_path = 'G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data'
clean_file = os.path.join(clean_path, '2023-03-10 Clean Names_query_df.csv')
query_df.to_csv(clean_file, index_label = False)

----